In [2]:
from typing import Any, Literal, TypeVar

import numpy as np
from bloqade.analysis.fidelity import FidelityAnalysis
from bloqade.types import Qubit
from kirin.dialects import ilist
from kirin.dialects.ilist import IList

from bloqade import squin
from bloqade.gemini.common.dialects import qubit
from bloqade.lanes.arch.gemini.logical import steane7_initialize
from bloqade.lanes.arch.gemini.physical import get_arch_spec
from bloqade.lanes.noise_model import generate_simple_noise_model
from bloqade.lanes.passes import ASAPPlacePass
from bloqade.lanes.pipeline import PhysicalPipeline
from bloqade.lanes.transform import MoveToSquinPhysical

kernel = squin.kernel.add(qubit)
kernel.run_pass = squin.kernel.run_pass

In [3]:
LogicalQubit = ilist.IList[Qubit, Literal[7]]

# TODO: add an image of the qubit layout

In [ ]:
def steane_slot_allocator():
    """Generates a qubit allocator for logical qubits.
    Tries to allocate logical qubits into the architecture in an efficient way to
    make parallelism in the logical gagdets as easily as possible in the move compiler.

    """
    # canonical slot order -- should the final one be 20?? or 18?
    slot_words = ilist.IList([0, 4, 8, 12, 16, 2, 6, 10, 14, 20])

    slots = IList(
        [
            IList([(0, word_id, site_id) for site_id in range(7)])
            for word_id in slot_words
        ]
    )

    print(f"slots: {slots}")

    @kernel
    def qalloc_slot(
        slot_index: int, theta: float, phi: float, lam: float
    ) -> LogicalQubit:
        def allocate_at(address: tuple[int, int, int]):
            return qubit.new_at(address[0], address[1], address[2])

        addresses = slots[slot_index]

        reg = ilist.map(allocate_at, addresses)

        steane7_initialize(theta, phi, lam, reg)

        return reg